# Payout Outcome Predictor — Exploration

This notebook is the **exploration** for the project: generate the synthetic data, look at it, and compare models. The production code lives in `generate_data.py`, `train.py`, and `app.py`.

Pipeline: **generate data → explore → encode & split → baseline tree → stronger models → pick the winner.**

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

## 2. Generate the synthetic dataset

Each feature is drawn from a realistic distribution; the `outcome` is the highest-scoring class (each class scored from its real-world drivers) plus noise, so it's probabilistic, not a hard rule. This is the same logic packaged in `generate_data.py`.

In [30]:
rng = np.random.default_rng(7)   # fixed seed -> reproducible
N = 8000
# order must match the score-stack order in the outcome cell
OUTCOME_CLASSES = ["SUCCESS", "FAIL_MOP", "FAIL_BANK_VALIDATION", "FAIL_NAME_MISMATCH", "FAIL_AMOUNT_LIMIT"]
print(f"{N} rows, {len(OUTCOME_CLASSES)} classes")

8000 rows, 5 classes


In [32]:
# ---- Features (16) ----
# Payout context
destination_region = rng.choice(["NAM","EU","LATAM","SSA","MENA","SEA","SA"], N, p=[.22,.20,.16,.10,.08,.14,.10])
payout_method      = rng.choice(["dlb_bank","wise","thunes_wallet","payoneer","paypal","ach","wire"], N, p=[.30,.18,.12,.16,.10,.08,.06])
amount_usd         = (rng.gamma(2.0, 160, N) + 20).round(2)
is_batch_payout    = rng.binomial(1, 0.60, N)

# Account history
account_age_days         = rng.integers(1, 2400, N)
prior_successful_payouts = rng.poisson(account_age_days / 90.0)
historical_failure_rate  = np.clip(rng.beta(1.5, 12, N) + (account_age_days < 120)*0.1, 0, 1).round(3)
days_since_last_payout   = rng.integers(0, 400, N)
top_rated_status         = np.clip((prior_successful_payouts > 15).astype(int) + (prior_successful_payouts > 40).astype(int), 0, 3)

# Method & verification
mop_age_days            = rng.integers(0, 1500, N)
mop_verified            = (rng.uniform(0, 1, N) < 0.88).astype(int)
bank_detail_age_days    = rng.integers(0, 2200, N)
p_invalid               = 0.03 + 0.07*(bank_detail_age_days > 730) + 0.04*(bank_detail_age_days > 1460) + 0.02*is_batch_payout
bank_account_valid      = (rng.uniform(0, 1, N) > p_invalid).astype(int)
name_has_special_chars  = rng.binomial(1, 0.14, N)
name_match_score        = np.clip(rng.beta(9, 1.2, N) - name_has_special_chars*rng.uniform(0.15, 0.5, N), 0, 1).round(3)
recent_bank_change_flag = rng.binomial(1, 0.12, N)
print("16 features generated.")

16 features generated.


In [34]:
# ---- Outcome: per-class score from drivers, highest (+ noise) wins ----
amount_z = (amount_usd - amount_usd.mean()) / amount_usd.std()
s_success = 2.6 + 0.04*np.minimum(prior_successful_payouts, 50) + 0.5*top_rated_status \
            - 3.0*historical_failure_rate + 0.0004*account_age_days
s_mop     = 3.0*(1-mop_verified) + 1.7*(mop_age_days < 30) + 1.6*is_batch_payout \
            + 1.0*recent_bank_change_flag + 0.6*(days_since_last_payout > 180)
s_bank    = 2.6*(1-bank_account_valid) + 1.3*(bank_detail_age_days > 730) + 0.9*is_batch_payout \
            + 0.5*(payout_method == "dlb_bank")
s_name    = 2.9*(name_match_score < 0.6) + 2.2*name_has_special_chars + 1.0*(name_match_score < 0.8)
s_amount  = 2.4*np.clip(amount_z, 0, None) + 3.2*(amount_usd > 900) + 2.0*(amount_usd > 1500)

scores  = np.vstack([s_success, s_mop, s_bank, s_name, s_amount]).T + rng.gumbel(0, 0.4, (N, 5))
outcome = np.array(OUTCOME_CLASSES)[scores.argmax(axis=1)]

In [ ]:
df = pd.DataFrame({
    "amount_usd": amount_usd, "payout_method": payout_method, "destination_region": destination_region,
    "is_batch_payout": is_batch_payout, "account_age_days": account_age_days,
    "prior_successful_payouts": prior_successful_payouts, "historical_failure_rate": historical_failure_rate,
    "days_since_last_payout": days_since_last_payout, "top_rated_status": top_rated_status,
    "mop_age_days": mop_age_days, "mop_verified": mop_verified, "recent_bank_change_flag": recent_bank_change_flag,
    "bank_account_valid": bank_account_valid, "bank_detail_age_days": bank_detail_age_days,
    "name_match_score": name_match_score, "name_has_special_chars": name_has_special_chars,
    "outcome": outcome,
})
df.to_csv("payouts.csv", index=False)
print("Shape:", df.shape, "| missing:", int(df.isna().sum().sum()))   # 16 features + outcome, 0 missing

## 3. Explore the data

Always inspect before modeling: class balance, a sample, and summary stats.

In [ ]:
print(df["outcome"].value_counts(normalize=True).round(3))
df.head()

In [ ]:
df.describe().round(2)

## 4. Prepare for modeling — encode & split

One-hot encode the two text columns (numbers, not fake order), then hold out 20% for honest evaluation. `stratify=y` keeps each class's proportion in both splits.

In [ ]:
TARGET = "outcome"
X = df.drop(columns=TARGET)
y = df[TARGET]
X_encoded = pd.get_dummies(X, columns=["payout_method", "destination_region"])
print("after one-hot:", X_encoded.shape[1], "columns")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape)

## 5. Baseline — Decision Tree

A single tree, depth-limited so it generalizes. Read **macro-F1** (not just accuracy) because the classes are imbalanced.

In [ ]:
tree = DecisionTreeClassifier(max_depth=12, random_state=42).fit(X_train, y_train)
pred = tree.predict(X_test)
print("Decision Tree | accuracy:", round(accuracy_score(y_test, pred), 3),
      "| macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))
print(classification_report(y_test, pred))

### Experiment (not the final model): the class-weight trap

How depth and class weighting trade off. **Lesson:** aggressive `class_weight="balanced"` on a *shallow* tree over-predicts rare classes and can tank accuracy. This is a teaching experiment — we do **not** use this configuration in production.

In [ ]:
for depth in [6, 12, None]:
    for cw in [None, "balanced"]:
        t = DecisionTreeClassifier(max_depth=depth, class_weight=cw, random_state=42).fit(X_train, y_train)
        p = t.predict(X_test)
        print(f"depth={str(depth):4s} class_weight={str(cw):8s} "
              f"acc={accuracy_score(y_test,p):.3f} macro-F1={f1_score(y_test,p,average='macro'):.3f}")

## 6. Stronger models — Random Forest & Gradient Boosting

A single tree is the weakest tree-based model. Ensembles usually do better; on tabular data, gradient-boosted trees are typically state of the art.

In [ ]:
tree = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42)
rf   = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=-1, random_state=42)
gbm  = HistGradientBoostingClassifier(max_depth=6, random_state=42)
for name, m in {"DecisionTree": tree, "RandomForest": rf, "GradientBoosting": gbm}.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    print(f"{name:18s} acc={accuracy_score(y_test,p):.3f} macro-F1={f1_score(y_test,p,average='macro'):.3f}")

## 7. Feature importance & confusion matrix

Do the features the model relies on match the drivers we designed? And which classes get confused?

In [ ]:
# Feature importance from the Random Forest
imp = sorted(zip(X_encoded.columns, rf.feature_importances_), key=lambda t: -t[1])
print("Top 10 features:")
for n, s in imp[:10]:
    print(f"  {n:34s} {s:.3f}")

# Confusion matrix for the chosen model (Gradient Boosting)
labels = sorted(y.unique())
cm = confusion_matrix(y_test, gbm.predict(X_test), labels=labels)
print("\nGradient Boosting confusion matrix (rows=actual, cols=predicted):")
print(pd.DataFrame(cm, index=labels, columns=labels))

## 8. Conclusion

Gradient boosting wins (best macro-F1) — the expected result for tabular data. The weak spot is `FAIL_MOP` / `FAIL_BANK_VALIDATION` recall: those look like SUCCESS until they fail, so the model is appropriately *less confident* there (visible in its probabilities). A neural net is **not** used — on small tabular data, boosted trees beat it.

**Productionized in:** `train.py` (trains the pipeline → `model.joblib`) and `app.py` (FastAPI `/predict`).